# Set 05 – Vollständige Pipeline mit Palmer Penguins

Dieses Notebook verbindet Online-Daten, Analyse, Datenaufteilung, numerische und kategoriale Vorverarbeitung, logistische Regression und Bewertung in einem Workflow.

Datensatz: Palmer Penguins von Allison Horst, Alison Hill und Kristen Gorman. Die Daten stehen unter CC0 und können damit auch kommerziell verwendet werden. Quelle: https://github.com/allisonhorst/palmerpenguins

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Daten online laden

Standardmäßig wird die offizielle CSV-Datei direkt aus dem Projekt geladen. Falls keine Internetverbindung verfügbar ist, wird die bereits in Set 03 enthaltene lokale Kopie verwendet.

In [ ]:
DATEN_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"

try:
    daten = pd.read_csv(DATEN_URL)
    quelle = "online"
except Exception as fehler:
    kandidaten = [
        Path("../set3-datenanalyse-und-visualisierung/daten/palmer_penguins.csv"),
        Path("set3-datenanalyse-und-visualisierung/daten/palmer_penguins.csv"),
    ]
    lokaler_pfad = next((pfad for pfad in kandidaten if pfad.exists()), None)
    if lokaler_pfad is None:
        raise FileNotFoundError("Online-Quelle und lokale Kopie sind nicht verfügbar.") from fehler
    daten = pd.read_csv(lokaler_pfad)
    quelle = f"lokal: {lokaler_pfad}"

print("Geladen:", quelle)
print("Form:", daten.shape)
display(daten.head())

## 2. Datenqualität und Zielvariable prüfen

species ist die Zielklasse. Es gibt drei Arten, also handelt es sich um Multiclass-Klassifikation.

In [ ]:
print("Datentypen:")
print(daten.dtypes)
print()
print("Fehlende Werte:")
print(daten.isna().sum())
print()
print("Klassen:")
print(daten["species"].value_counts())

## 3. Daten visualisieren

Schnabellänge und Schnabeltiefe trennen die Arten bereits teilweise. Überlappungen zeigen aber, warum mehrere Merkmale hilfreich sind.

In [ ]:
farben = {"Adelie": "#4C78A8", "Chinstrap": "#E45756", "Gentoo": "#54A24B"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for art, gruppe in daten.groupby("species"):
    axes[0].scatter(
        gruppe["bill_length_mm"], gruppe["bill_depth_mm"],
        label=art, alpha=0.7, color=farben[art]
    )
axes[0].set_xlabel("Schnabellänge in mm")
axes[0].set_ylabel("Schnabeltiefe in mm")
axes[0].set_title("Körpermaße nach Art")
axes[0].legend()

daten["species"].value_counts().plot.bar(ax=axes[1], color=[farben[name] for name in daten["species"].value_counts().index])
axes[1].set_title("Anzahl pro Art")
axes[1].set_xlabel("Art")
axes[1].set_ylabel("Anzahl")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. X und y bilden und aufteilen

Die Art wird nicht als Eingabemerkmal verwendet. stratify erhält die Artenanteile im Testdatensatz.

In [ ]:
merkmale = [
    "island",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "sex",
]
X = daten[merkmale]
y = daten["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Training:", X_train.shape)
print("Test:", X_test.shape)

## 5. Vorverarbeitung definieren

Numerische Spalten: Median-Imputation und StandardScaler.

Kategoriale Spalten: häufigste Kategorie und OneHotEncoder. Unbekannte Kategorien werden ignoriert.

In [ ]:
numerische_spalten = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]
kategoriale_spalten = ["island", "sex"]

numerische_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
kategoriale_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

vorverarbeitung = ColumnTransformer([
    ("numerisch", numerische_pipeline, numerische_spalten),
    ("kategorial", kategoriale_pipeline, kategoriale_spalten),
])

## 6. Vollständige Pipeline

Der letzte Schritt ist LogisticRegression. Bei drei Klassen lernt scikit-learn mehrere lineare Entscheidungen. fit() lernt Vorverarbeitung und Modell ausschließlich aus den Trainingsdaten.

In [ ]:
pipeline = Pipeline([
    ("vorverarbeitung", vorverarbeitung),
    ("modell", LogisticRegression(max_iter=2000)),
])
pipeline.fit(X_train, y_train)

## 7. Transformierte Merkmale nachvollziehen

ColumnTransformer erzeugt aus sechs Rohspalten mehrere numerische Modellspalten. Die Testdaten werden nur transformiert.

In [ ]:
transformer = pipeline.named_steps["vorverarbeitung"]
feature_namen = transformer.get_feature_names_out()
X_test_transformiert = transformer.transform(X_test)

print("Rohspalten:", X_test.shape[1])
print("Modellspalten:", X_test_transformiert.shape[1])
print(feature_namen)

## 8. Multiclass-Ergebnisse

Der Bericht zeigt Precision, Recall und F1 für jede Art. macro avg gewichtet jede Art gleich; weighted avg berücksichtigt ihre Häufigkeit.

In [ ]:
y_pred = pipeline.predict(X_test)
bericht = pd.DataFrame(
    classification_report(y_test, y_pred, output_dict=True, zero_division=0)
).T
display(bericht.round(3))

## 9. Konfusionsmatrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", xticks_rotation=25
)
plt.title("Palmer Penguins – Testdaten")
plt.tight_layout()
plt.show()

## 10. Wahrscheinlichkeiten für neue Rohdaten

predict_proba() liefert für jede Art eine Wahrscheinlichkeit. Die neue Tabelle darf dieselben Rohspalten wie beim Training enthalten; Imputation, Skalierung und Encoding passieren automatisch.

In [ ]:
neuer_pinguin = pd.DataFrame({
    "island": ["Biscoe"],
    "bill_length_mm": [47.0],
    "bill_depth_mm": [15.0],
    "flipper_length_mm": [218],
    "body_mass_g": [5050],
    "sex": ["male"],
})

wahrscheinlichkeiten = pipeline.predict_proba(neuer_pinguin)
proba_tabelle = pd.DataFrame(
    wahrscheinlichkeiten,
    columns=pipeline.named_steps["modell"].classes_,
)
display(proba_tabelle.round(3))
print("Vorhergesagte Art:", pipeline.predict(neuer_pinguin)[0])

## Zusammenfassung

Die Pipeline hält alle lernenden Schritte zusammen. Für echte Projekte müssten zusätzlich Datenherkunft, Repräsentativität, Messfehler, fachliche Kosten unterschiedlicher Fehler und die langfristige Überwachung des Modells betrachtet werden.